# Checkpoint 2 — dish classifier baseline

Checkpoint 1 predicted **calories** from a food name and a serving size. The committed calorie table has one row per dish, so that model never sees a photo, and at test time a new dish name has no row to look up.

Checkpoint 2 changes the trained model to the step a photo actually needs: **name the dish**. Calories stay a table lookup after the class is predicted. The class list is the 27 Food-101 dishes that already have a calorie row, not all 101 classes.

The trained model is L2 logistic regression on a frozen ImageNet ResNet18 embedding. Training code: `scripts/checkpoint2_baseline.py`. Re-run with:

```bash
.venv/bin/python scripts/checkpoint2_baseline.py
```

Features are cached, so a second run only refits the classifier.

## 1. Preprocessing pipeline

Raw input is a Food-101 JPEG (max side 512). The calorie table is `scripts/food_calories.json`.

Cleaning that we actually applied:

- Kept a class only when its folder name appears in the calorie table. All 27 names match a Food-101 class. The other 74 Food-101 classes are dropped because a prediction there cannot be turned into calories yet.
- The calorie table has no duplicate names and no missing calories. `onion_rings` has `grams: null`. Serving size is not a model input, so that missing value is not imputed and does not affect training. It still has to be filled before we report a calorie number for onion rings.
- Images are opened as RGB. Food-101 has no missing-pixel problem to impute.
- The ResNet18 ImageNet transform resizes the short side to 256, center-crops 224×224, scales pixels to [0, 1], then normalizes with ImageNet mean `[0.485, 0.456, 0.406]` and std `[0.229, 0.224, 0.225]`. Those constants are fixed. They are not estimated from our train or test images.
- The frozen ResNet18 (eval mode, so batch-norm uses ImageNet running stats) replaces its classification layer with an identity. Its global-average-pool output is the feature vector.

No one-hot encoding of the food name. The name is the label, not a feature. That is the change from Checkpoint 1, where `food_item` was an input column.

## 2. Model input

One sample is one photograph of a dish.

- Tensor into the backbone: `(3, 224, 224)` — three RGB channels, 224×224.
- Matrix into the trained classifier: `X_train` is `(20250, 512)`, `y_train` is `(20250,)`. `X_test` is `(6750, 512)`, `y_test` is `(6750,)`. Each row of `X` is the 512-d ResNet18 embedding. Each entry of `y` is an integer class id in `0..26`.

## 3. Train/test split

We use Food-101's official split, then drop classes that are not in the calorie table.

- Train: 750 images per class, 20,250 total.
- Test: 250 images per class, 6,750 total.
- The two lists are disjoint, so no photo is in both sets. Counts are equal per class, so the split is stratified by construction.
- Normalization constants and the backbone weights come from ImageNet, not from these images.
- The logistic regression is fit on train embeddings only. `C=1.0` was chosen before looking at the test set. There is no validation split yet, because nothing was tuned. Checkpoint 3 should carve a stratified validation set out of the official **train** images and leave this test set untouched.

`model/data/preprocessing.py` would make a fresh 80/20 split from the class folders. That would mix official test photos into training. This baseline does not do that.

## 4. Baseline model

This is 27-way classification: one photo, one dish, classes mutually exclusive.

Logistic regression (softmax over 27 classes) is the linear baseline on top of a pretrained vision embedding. A linear probe answers whether the embedding already separates the dishes before we spend Checkpoint 3 fine-tuning the network in `model/train.py`.

## 5. Training code and hyperparameters

Fit happens in `fit_baseline` inside `scripts/checkpoint2_baseline.py`:

```python
clf = LogisticRegression(
    C=1.0,
    l1_ratio=0.0,
    solver="lbfgs",
    max_iter=1000,
    random_state=42,
)
clf.fit(x_train, y_train)
```

- `C=1.0` is the inverse of the regularization strength. Smaller `C` means a stronger penalty.
- `l1_ratio=0.0` makes that penalty pure L2 (no sparse L1 term).
- `solver="lbfgs"` is the optimizer for the multinomial loss. It converged in 800 iterations, under the `max_iter=1000` cap.
- `random_state=42` fixes the solver's tie-breaking so the fit is repeatable.

The ResNet18 weights are not updated.

## 6. Metrics

- **Accuracy.** Classes are balanced (250 test images each), so accuracy is the fraction of photos whose top prediction is the right dish.
- **Macro F1.** Averages per-class F1 with equal weight, so a dish the model always misses cannot hide behind the easy classes. Precision and recall per dish are in `results/checkpoint2/classification_report.txt`.
- **Confusion matrix.** Shows which dishes are swapped, which accuracy cannot.

Chance accuracy, from always predicting the most common training class, is **3.7%** (1/27). RMSE and ROC-AUC are the wrong headlines: the decision is a single class label, not a calorie number and not a threshold on one score.

## 7. Test results

Evaluated on the official test split, not the training images.

| | |
|---|---|
| Majority-class accuracy | 0.037 |
| Train accuracy | 0.874 |
| **Test accuracy** | **0.712** |
| **Test macro F1** | **0.711** |

![Row-normalized confusion matrix on the test set](../results/checkpoint2/confusion_matrix.png)

![Per-class test recall](../results/checkpoint2/per_class_recall.png)

![High-confidence mistakes](../results/checkpoint2/high_confidence_mistakes.png)

## 8. Interpretation

71.2% test accuracy is a real classifier: about 19× the 3.7% chance line, on photos it did not train on. Macro F1 matches accuracy because every class has the same support, and no class has collapsed.

Train accuracy is 87.4%, 16 points above test. That is mild overfitting of the linear layer. It is not underfitting: both numbers are far above chance, and the training fit is strong. The gap is a reason to tune `C` on a validation split and then fine-tune the backbone, not a reason to throw out the embedding.

The easy dishes have a distinctive shape: pizza (recall 0.86), onion rings (0.85), deviled eggs (0.84), french fries (0.83). The hard ones are near-duplicates:

- ravioli is the weakest class (recall 0.50). 23 of 250 lasagna photos are called ravioli, and 17 ravioli photos are called lasagna.
- pulled-pork sandwich and hamburger swap both ways (31 and 29 test photos).
- pork chop and baby back ribs swap (24 and 17).
- samosa is called spring rolls 17 times. spaghetti bolognese is called pad thai 19 times.

The high-confidence mistakes are the same kind of error, plus plates that contain two foods. One pulled-pork image is mostly onion rings, and the model predicts onion rings. The label is a single dish, but the photo is not. A frozen embedding will keep confusing those pairs until the network can be trained to use the differences (bun vs. sandwich filling, pasta shape, rib vs. chop).

## 9. Next steps, before the final project

1. **Fine-tune the network** with `model/train.py` instead of freezing it. That trainer already warms up the new classification head, then unfreezes the backbone at a lower learning rate. The dishes that share a look (pasta, sandwiches, fried meat) are the ones a linear probe cannot separate.
2. **Tune hyperparameters on a validation split cut from the official train set only.** First candidate is `C` for this linear model; after fine-tuning, learning rate and how many epochs the backbone stays frozen. The official test set stays untouched until the final number.

After the class is predicted, attach calories from `food_calories.json`. Fill the missing onion-rings serving size before that lookup, and expect calorie error on photos that contain more than one dish.

In [ ]:
import json
from pathlib import Path

summary = json.loads(Path("../results/checkpoint2/metrics.json").read_text())
print("X_train", tuple(summary["x_train_shape"]), "y_train", tuple(summary["y_train_shape"]))
print("X_test ", tuple(summary["x_test_shape"]), "y_test ", tuple(summary["y_test_shape"]))
print(f"train acc {summary['train_accuracy']:.3f}")
print(f"test acc  {summary['test_accuracy']:.3f}")
print(f"test macro F1 {summary['test_macro_f1']:.3f}")
print("top confusions:")
for row in summary["top_confusions"]:
    print(f"  {row['count']:3d}  {row['true']} -> {row['predicted']}")